In [16]:
pip install biopython


In [17]:
!pip install pandas chromadb sentence-transformers transformers

In [18]:
from Bio import Entrez
import pandas as pd


In [19]:
from Bio import Entrez

In [20]:
Entrez.email = "anjususan.thomas@student.uq.edu.au"

In [21]:
query = "parkinson GWAS"
max_results = 100

search_handle = Entrez.esearch(
    db="pubmed",
    term=query,
    retmax=max_results
)

search_results = Entrez.read(search_handle)
pmids = search_results["IdList"]

print(pmids)

['42245814', '42245037', '42227498', '42216962', '42213730', '42209523', '42200344', '42192339', '42177729', '42168222', '42166149', '42163669', '42163007', '42160433', '42159407', '42152389', '42132746', '42129508', '42116350', '42116305', '42079086', '42078358', '42069741', '42057133', '42051091', '42020718', '42009659', '41991085', '41981742', '41966729', '41961677', '41930763', '41929179', '41929081', '41926874', '41925436', '41916085', '41908051', '41889906', '41872227', '41855783', '41822813', '41820575', '41810385', '41808632', '41801886', '41791140', '41776125', '41761858', '41761254', '41757182', '41733927', '41735219', '41728547', '41724112', '41721963', '41710648', '41699864', '41684331', '41674635', '41672153', '41667488', '41651665', '41646826', '41646756', '41646738', '41638368', '41635116', '41619353', '41607670', '41588886', '41584517', '41562493', '41556177', '41545942', '41543776', '41542687', '41540060', '41518215', '41504309', '41503462', '41497575', '41469746', '41

In [22]:
papers = []

for pmid in pmids:
    fetch_handle = Entrez.efetch(
        db="pubmed",
        id=pmid,
        rettype="abstract",
        retmode="text"
    )

    abstract_text = fetch_handle.read()

    papers.append({
        "pmid": pmid,
        "abstract": abstract_text
    })

df = pd.DataFrame(papers)
df.head()

,pmid,abstract
0,42245814,1. Res Sq [Preprint]. 2026 May 28:rs.3.rs-9622...
1,42245037,1. medRxiv [Preprint]. 2026 May 25:2026.05.23....
2,42227498,1. Comb Chem High Throughput Screen. 2026 May ...
3,42216962,1. Biometrics. 2026 Apr 9;82(2):ujag090. doi: ...
4,42213730,1. PLoS Comput Biol. 2026 May 29;22(5):e101432...


In [23]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = df["abstract"].tolist()

embeddings = embedding_model.encode(texts)

print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(100, 384)


In [24]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = df["abstract"].tolist()

embeddings = embedding_model.encode(texts)

print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(100, 384)


In [25]:
!pip install chromadb
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="pubmed_abstracts"
)

for i, row in df.iterrows():
    collection.add(
        ids=[str(row["pmid"])],
        documents=[row["abstract"]],
        embeddings=[embeddings[i].tolist()],
        metadatas=[{"pmid": str(row["pmid"])}]
    )

print("Stored abstracts in ChromaDB")

Stored abstracts in ChromaDB


In [26]:
question = "What genes are associated with parkinson risk ?"

question_embedding = embedding_model.encode([question])[0]

results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=3
)

results

{'ids': [['41331295', '42245037', '42168222']],
 'embeddings': None,
 'documents': [["1. NPJ Parkinsons Dis. 2025 Dec 2;11(1):348. doi: 10.1038/s41531-025-01180-z.\n\nTMEM175, SCARB2 and CTSB associations with Parkinson's disease risk across \npopulations.\n\nSun W(1), Schulte C(1), Gasser T(2), Tan M(3); Global Parkinson’s Genetic \nProgram (GP2).\n\nCollaborators: Pihlstrøm L, Chana P, Song Y, Bandres-Ciga S, Blauwendraat C, \nSingleton A, Nalls MA, Leonard H, Rizig M, Iwaki H, Rieder C, Mata IF, Okubadejo \nN, Gatto EM, Kauffman M, Shepherd CE, Khachatryan S, Tavadyan Z, Hunter J, Kumar \nK, Ellis M, Rentería ME, Koks S, Zimprich A, Tumas V, Camargos S, Fon EA, Fon T, \nMonchi O, Galleguillos BP, Olguin P, Miranda M, Bustamante ML, Tang B, Shang H, \nGuo J, Chan P, Luo W, Arboleda G, Orozco J, Del Rio MJ, Hernandez A, Salama M, \nKamel WA, Zewde YZ, Brice A, Corvol JC, Westenberger A, Klein C, Vollstedt EJ, \nMadoev H, Trinh J, Junker J, Lohmann K, Illarionova A, Mollenhauer B, Hopf

In [27]:
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print("PMID:", meta["pmid"])
    print(doc[:1000])
    print("-" * 80)

PMID: 41331295
1. NPJ Parkinsons Dis. 2025 Dec 2;11(1):348. doi: 10.1038/s41531-025-01180-z.

TMEM175, SCARB2 and CTSB associations with Parkinson's disease risk across 
populations.

Sun W(1), Schulte C(1), Gasser T(2), Tan M(3); Global Parkinson’s Genetic 
Program (GP2).

Collaborators: Pihlstrøm L, Chana P, Song Y, Bandres-Ciga S, Blauwendraat C, 
Singleton A, Nalls MA, Leonard H, Rizig M, Iwaki H, Rieder C, Mata IF, Okubadejo 
N, Gatto EM, Kauffman M, Shepherd CE, Khachatryan S, Tavadyan Z, Hunter J, Kumar 
K, Ellis M, Rentería ME, Koks S, Zimprich A, Tumas V, Camargos S, Fon EA, Fon T, 
Monchi O, Galleguillos BP, Olguin P, Miranda M, Bustamante ML, Tang B, Shang H, 
Guo J, Chan P, Luo W, Arboleda G, Orozco J, Del Rio MJ, Hernandez A, Salama M, 
Kamel WA, Zewde YZ, Brice A, Corvol JC, Westenberger A, Klein C, Vollstedt EJ, 
Madoev H, Trinh J, Junker J, Lohmann K, Illarionova A, Mollenhauer B, Hopfner F, 
Höglinger G, Lange LM, Sharma M, Groppa S, Fang ZH, Akpalu A, Xiromerisiou G, 

In [28]:
retrieved_context = ""

for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    retrieved_context += f"PMID: {meta['pmid']}\n{doc}\n\n"

prompt = f"""
You are a biomedical research assistant.

Use only the PubMed abstracts below to answer the question.
Cite evidence using PMID numbers.
If the abstracts do not contain enough evidence, say so clearly.

Question:
{question}

PubMed abstracts:
{retrieved_context}

Answer:
"""

print(prompt[:3000])


You are a biomedical research assistant.

Use only the PubMed abstracts below to answer the question.
Cite evidence using PMID numbers.
If the abstracts do not contain enough evidence, say so clearly.

Question:
What genes are associated with parkinson risk ?

PubMed abstracts:
PMID: 41331295
1. NPJ Parkinsons Dis. 2025 Dec 2;11(1):348. doi: 10.1038/s41531-025-01180-z.

TMEM175, SCARB2 and CTSB associations with Parkinson's disease risk across 
populations.

Sun W(1), Schulte C(1), Gasser T(2), Tan M(3); Global Parkinson’s Genetic 
Program (GP2).

Collaborators: Pihlstrøm L, Chana P, Song Y, Bandres-Ciga S, Blauwendraat C, 
Singleton A, Nalls MA, Leonard H, Rizig M, Iwaki H, Rieder C, Mata IF, Okubadejo 
N, Gatto EM, Kauffman M, Shepherd CE, Khachatryan S, Tavadyan Z, Hunter J, Kumar 
K, Ellis M, Rentería ME, Koks S, Zimprich A, Tumas V, Camargos S, Fon EA, Fon T, 
Monchi O, Galleguillos BP, Olguin P, Miranda M, Bustamante ML, Tang B, Shang H, 
Guo J, Chan P, Luo W, Arboleda G, Orozco